# Setup data

In [3]:
import torch
from scipy.special import softmax

def load_params(model_path, weights_only=False):
    # 1) Load parameters
    model_data = torch.load(model_path, map_location=torch.device('cpu'), weights_only=weights_only)
    theta = model_data['theta']          # shape: (n_people, k)
    d = model_data['d']                  # shape: (n_items,  k)
    p = model_data['phi']

    print("--- Initial Loaded Data ---")
    print(f"Original theta shape: {theta.shape}")
    print(f"Original 'd' matrix shape: {d.shape}\n")

    d_numpy = d.detach().cpu().numpy()
    theta_numpy = theta.detach().cpu().numpy()
    p_numpy = p.detach().cpu().numpy()
    w_numpy = softmax(p_numpy, axis=1)
    return d_numpy, theta_numpy, p_numpy, w_numpy

In [4]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append('..')
import style
# Concatenate math and gsm outputs for theta, a, b

d, theta, p, w = load_params('../result/lada-fitting-joint/lada_joint_k3_lsat_qa.pt', weights_only=False)

resmat = pd.read_pickle('../data-reeval-multi/resmat.pkl')

conv = ['lsat_qa']

conv_mask = resmat.loc[:, resmat.columns.get_level_values("scenario").isin(conv)]

conv_questions = conv_mask.columns.get_level_values('input.text').tolist()

answers_path = Path('../data-processing/lsat_qa/lsat_qa_result.pkl')
if not answers_path.exists():
    raise FileNotFoundError(f'Expected answer key at {answers_path} was not found. Run the assembly script first.')
answer_df = pd.read_pickle(answers_path)
answer_frame = answer_df.loc['answer'].rename('answer').reset_index()
type_frame = answer_df.loc['type'].rename('type').reset_index()
answer_frame['scenario'] = answer_frame['scenario'].astype(str).str.lower()
type_frame['scenario'] = type_frame['scenario'].astype(str).str.lower()
merged_answers = answer_frame.merge(
    type_frame[['scenario', 'input.text', 'type']],
    on=['scenario', 'input.text'],
    how='left'
 )
answer_map = merged_answers[merged_answers['scenario'] == 'lsat_qa'].set_index('input.text')[['answer', 'type']]
answers_aligned = answer_map.reindex(conv_questions)
if answers_aligned['answer'].isna().any() or answers_aligned['type'].isna().any():
    missing_answers = answers_aligned[answers_aligned['answer'].isna()].index.tolist()
    missing_types = answers_aligned[answers_aligned['type'].isna()].index.tolist()
    raise ValueError(
        'Missing answers or types for prompts. '
        f'Missing answers: {missing_answers[:5]} | Missing types: {missing_types[:5]}'
    )

theta_df = pd.DataFrame(theta, columns=[f'Factor_{i+1}' for i in range(theta.shape[1])], index=resmat.index)
w_df = pd.DataFrame(w, columns=[f'Factor_{i+1}' for i in range(w.shape[1])])
w_df['question'] = conv_questions
w_df['answer'] = answers_aligned['answer'].values
w_df['type'] = answers_aligned['type'].values

models_answered = conv_mask.dropna(how='all').index.tolist()
accuracy = (conv_mask.sum(axis=1) / conv_mask.notnull().sum(axis=1)).loc[models_answered]
theta_irt = pd.read_csv('../result/irt-fitting/calibration_result_theta_lsat_qa.csv')
theta_df = theta_df.loc[models_answered]
theta_df['accuracy'] = accuracy
theta_df['irt'] = theta_irt.values

--- Initial Loaded Data ---
Original theta shape: torch.Size([183, 3])
Original 'd' matrix shape: torch.Size([454])



In [24]:
import os

os.makedirs('../output/lada', exist_ok=True)

export_cols = ['Factor_1', 'Factor_2', 'Factor_3', 'question', 'answer', 'type']
available_cols = [col for col in export_cols if col in w_df.columns]
w_export = w_df[available_cols]
w_export.to_csv(f'../output/lada/lsat_qa_lada_3k.csv', index=False)
w_export.sort_values(by=['Factor_1'], ascending=False).to_csv(f'../output/lada/lsat_qa_lada_3k_f1_desc.csv', index=False)
w_export.sort_values(by=['Factor_2'], ascending=False).to_csv(f'../output/lada/lsat_qa_lada_3k_f2_desc.csv', index=False)
w_export.sort_values(by=['Factor_3'], ascending=False).to_csv(f'../output/lada/lsat_qa_lada_3k_f3_desc.csv', index=False)

# Analysis

In [8]:
import pandas as pd

df = pd.read_csv(f'../output/lada/lsat_qa_lada_3k.csv')

In [23]:
df.sort_values(ascending=False, by='Factor_1')

,Factor_1,Factor_2,Factor_3,question,answer,type
35,0.991711,0.004302,0.003987,"A company's six vehicles—a hatchback, a limous...","the pickup, the roadster, the hatchback",ar
171,0.990973,0.006177,0.002850,A television programming director is schedulin...,"Roamin', Sundown, Waterloo, Terry, Generations",ar
382,0.990857,0.005216,0.003927,"Of the eight students—George, Helen, Irving, K...",Irving gives a report on Monday.,ar
355,0.989285,0.004644,0.006071,"Four employees—Jackson, Larabee, Paulson, and ...",Jackson: Z; Larabee: X; Paulson: W; Torillo: Y,ar
406,0.989067,0.005820,0.005113,"Seven workers—Quinn, Ruiz, Smith, Taylor, Verm...",Neither Quinn nor Taylor is selected.,ar
...,...,...,...,...,...,...
316,0.005531,0.853440,0.141029,"Exactly six members of a skydiving team—Larue,...",Pei dives from the plane fourth.,ar
371,0.005487,0.734801,0.259712,"In the Lifestyle, Metro, and Sports sections o...",Exactly one photograph in the Sports section i...,ar
295,0.005299,0.783788,0.210912,Exactly five movies are showing at the reperto...,"the western, the horror film",ar
133,0.005032,0.009006,0.985962,A record producer is planning the contents of ...,Reciprocity is the last piece on the CD.,ar
